# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Samra-ca/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: Score pages for review when they are stale and still visible. The highest priority is stale pages with traffic, then stale pages with weak position, then low CTR despite position.

Reason codes:
- `stale_visible`: page is overdue for refresh and still has traffic.
- `stale_position`: page is stale and ranking position is weaker than expected.
- `ctr_position_mismatch`: page has low CTR despite a decent position.
- `baseline`: fallback review candidate.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

cwd = Path.cwd()
if cwd.name == 'notebooks' and cwd.parent.name == 'work':
    raw_path = cwd.parent.parent / 'data' / 'raw' / 'content_refresh_anonymized.csv'
    output_path = cwd.parent / 'outputs' / 'baseline_action_score.csv'
else:
    raw_path = cwd / 'data' / 'raw' / 'content_refresh_anonymized.csv'
    output_path = cwd / 'work' / 'outputs' / 'baseline_action_score.csv'

df = pd.read_csv(raw_path)

df['is_declining_label'] = df['trend_direction'].eq('down')
df['stale'] = df['days_since_last_update'] >= 60
df['good_position'] = df['avg_position'].between(1, 10)
df['low_ctr'] = df['ctr'] <= 0.5
df['ctr_position_signal'] = df['low_ctr'] & df['good_position']

print('Signal 1: staleness behind refresh flags')
stale_table = pd.crosstab(df['stale'], df['trend_direction'], margins=True)
print(stale_table)
print()
stale_decline = df.loc[df['stale'], 'is_declining_label'].mean()
fresh_decline = df.loc[~df['stale'], 'is_declining_label'].mean()
print(f'Decline rate if stale: {stale_decline:.3f}')
print(f'Decline rate if fresh: {fresh_decline:.3f}')
print('Verdict: CONFIRMED — stale pages are more likely to be declining and deserve review.')

print('\nSignal 2: CTR fix logic behind low CTR vs position')
ctr_table = pd.crosstab(df['ctr_position_signal'], df['trend_direction'], margins=True)
print(ctr_table)
print()
ctr_decline = df.loc[df['ctr_position_signal'], 'is_declining_label'].mean()
print(f'Decline rate for low-CTR strong-position pages: {ctr_decline:.3f}')
print('Verdict: MIXED — the CTR/position slice identifies pages worth checking, but not all are currently declining.')


Signal 1: staleness behind refresh flags
trend_direction   down  flat   new  stable    up    All
stale                                                  
False            10544   898  2128    3834  3199  20603
True              5718   254   108    2128  1189   9397
All              16262  1152  2236    5962  4388  30000

Decline rate if stale: 0.608
Decline rate if fresh: 0.512
Verdict: CONFIRMED — stale pages are more likely to be declining and deserve review.

Signal 2: CTR fix logic behind low CTR vs position
trend_direction       down  flat   new  stable    up    All
ctr_position_signal                                        
False                10199   348  1810    4058  3267  19682
True                  6063   804   426    1904  1121  10318
All                  16262  1152  2236    5962  4388  30000

Decline rate for low-CTR strong-position pages: 0.588
Verdict: MIXED — the CTR/position slice identifies pages worth checking, but not all are currently declining.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# Build the ranked queue and write output
rule_features = [
    'impressions_90d',
    'clicks_90d',
    'avg_position',
    'ctr',
    'days_since_last_update',
    'content_age_days'
]

score = (
    (df['impressions_90d'] >= 100).astype(int) *
    (df['days_since_last_update'] >= 60).astype(int) *
    (df['avg_position'].replace(0, np.nan).fillna(999).rpow(1))
)

# Build a transparent score that prioritizes stale pages with some visibility and a weaker position.
df['baseline_score'] = (
    df['impressions_90d'] / 1000 +
    (df['days_since_last_update'] >= 60).astype(int) * 2 +
    (df['avg_position'].replace(0, np.nan).fillna(999)) / 20
)

def reason_code(row):
    if row['days_since_last_update'] >= 180 and row['impressions_90d'] >= 200:
        return 'stale_visible'
    if row['days_since_last_update'] >= 60 and row['avg_position'] <= 30:
        return 'stale_position'
    if row['ctr'] <= 0.5 and row['avg_position'] <= 10:
        return 'ctr_position_mismatch'
    return 'baseline'


df['reason_code'] = df.apply(reason_code, axis=1)
df['action'] = 'review_page'

df = df.sort_values(by='baseline_score', ascending=False)
output_path = Path('work/outputs/baseline_action_score.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
df[['content_id', 'client_id', 'baseline_score', 'reason_code', 'action']].to_csv(output_path, index=False)
print('Wrote', output_path)
print('Top 10 rows:')
print(df[['content_id', 'baseline_score', 'reason_code', 'action']].head(10).to_string(index=False))


Wrote work\outputs\baseline_action_score.csv
Top 10 rows:
          content_id  baseline_score           reason_code      action
content_5fe46e04994d         519.925        stale_position review_page
content_aaef01a50def         517.379 ctr_position_mismatch review_page
content_8c19996aa890         509.377 ctr_position_mismatch review_page
content_2cb567c3c89b         498.837              baseline review_page
content_4c36c775b818         463.218 ctr_position_mismatch review_page
content_2dba2b1f9536         446.829        stale_position review_page
content_1a9e894be2e2         416.380 ctr_position_mismatch review_page
content_2c2606c5d176         349.609        stale_position review_page
content_db5989a78dd3         345.381 ctr_position_mismatch review_page
content_44e481c8f55b         312.764              baseline review_page


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
top20 = df[['content_id', 'client_id', 'baseline_score', 'reason_code', 'action', 'trend_direction', 'impressions_90d', 'avg_position', 'ctr', 'days_since_last_update']].head(20)
print('Top 20 review:')
for idx, row in top20.iterrows():
    print(f"- action={row['action']}, reason={row['reason_code']}, score={row['baseline_score']:.3f}, "
          f"why: stale {row['days_since_last_update']} days + pos {row['avg_position']} + impressions {row['impressions_90d']}.")
    print(f"  what would make it wrong: if position is stale due to tracking gaps, or the page already has no real visibility despite impressions.")


Top 20 review:
- action=review_page, reason=stale_position, score=519.925, why: stale 104 days + pos 4.2 + impressions 517715.
  what would make it wrong: if position is stale due to tracking gaps, or the page already has no real visibility despite impressions.
- action=review_page, reason=ctr_position_mismatch, score=517.379, why: stale 22 days + pos 5.4 + impressions 517109.
  what would make it wrong: if position is stale due to tracking gaps, or the page already has no real visibility despite impressions.
- action=review_page, reason=ctr_position_mismatch, score=509.377, why: stale 20 days + pos 2.5 + impressions 509252.
  what would make it wrong: if position is stale due to tracking gaps, or the page already has no real visibility despite impressions.
- action=review_page, reason=baseline, score=498.837, why: stale 48 days + pos 22.2 + impressions 497727.
  what would make it wrong: if position is stale due to tracking gaps, or the page already has no real visibility despite impr

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
print('Weak picks and leakage check:')
weak = top20[(top20['trend_direction'] == 'stable') | (top20['ctr'] > 1.0)]
print(weak[['content_id', 'reason_code', 'trend_direction', 'ctr', 'avg_position']].to_string(index=False))
print('\nWeak picks:')
for idx, row in weak.head(5).iterrows():
    print(f"- {row['content_id']}: reason={row['reason_code']}, trend={row['trend_direction']}, possible wrong if the page is stable or has healthy engagement.")

print('\nLeakage check: confirming no future-window or product-flag fields used.')
print('Used fields:', ['impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'content_age_days'])
print('Excluded label-derived fields: trend_direction, trend_pct')


Weak picks and leakage check:
          content_id           reason_code trend_direction  ctr  avg_position
content_aaef01a50def ctr_position_mismatch          stable 0.25           5.4
content_2dba2b1f9536        stale_position          stable 0.21          27.9
content_44e481c8f55b              baseline          stable 0.65           1.4
content_36ff89c8214e        stale_position          stable 0.05           7.3
content_b28d1efd668f        stale_position          stable 0.06          26.2
content_8e7ba84a972b              baseline          stable 0.92           4.8
content_89e84d699e9e              baseline          stable 0.89           4.8
content_aa4baf490b43 ctr_position_mismatch          stable 0.50           5.9

Weak picks:
- content_aaef01a50def: reason=ctr_position_mismatch, trend=stable, possible wrong if the page is stable or has healthy engagement.
- content_2dba2b1f9536: reason=stale_position, trend=stable, possible wrong if the page is stable or has healthy engagement

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.